---

# Statistics Explore 2026 - Disaster Type and Severity Classification

*Dual-head disaster type and severity classification with DINOv3 ViT-7B/16.*

---

## Team

- Farrel (Modelling)
- Member 2
- Member 3

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Exploratory Data Analysis**](#3)
4. [**Validation Design**](#4)
5. [**Preprocessing**](#5)
6. [**Modelling**](#6)
7. [**Training**](#7)
8. [**Evaluation**](#8)
9. [**Inference and Submission**](#9)
10. [**Conclusion**](#10)


---

# Introduction <a name="1"></a>

---

## Overview

*The Statistics Explore 2026 image track asks for two labels per photograph of a natural disaster: the **disaster type** (`jenis`) among flood, earthquake and fire, and the **damage severity** (`kerusakan`) among light, moderate and severe. The two axes combine into nine classes, but the submission file scores them as two separate rows per image, so the model is built with that structure in mind rather than as a flat nine-way classifier.*

*The training set contains 17,482 web-scraped photographs organised as `TRAIN/<jenis>/<kerusakan>/`, and the test set contains 450 unlabelled images named `1.jpg` through `450.jpg`. The scrape is heterogeneous: 3,909 distinct resolutions, four container formats and four colour modes appear, so the loading path has to normalise aggressively before any tensor is produced.*

## Aim

*This notebook fine-tunes **DINOv3 ViT-7B/16** into a dual-head classifier that predicts `jenis` and `kerusakan` jointly, and writes a submission in the exact semicolon-separated two-rows-per-image format the competition expects.*

*A secondary aim is to make the validation number trustworthy. The dataset contains a large population of near-duplicate frames, and a naive random split leaks them across the train and validation sides, which inflates the reported score well above what the leaderboard will return. Section 4 builds a grouped split that removes that leak.*

## Metric

***Micro F1-Score** aggregates true positives, false positives and false negatives over all classes before computing the ratio, rather than averaging per-class scores.*

$$\text{F1}_{\text{micro}} = \frac{2 \cdot \text{TP}}{2 \cdot \text{TP} + \text{FP} + \text{FN}}$$

*The submission has exactly 900 rows: 450 images times two targets. Every row carries exactly one predicted label and exactly one true label, so each row contributes either one true positive, or one false positive plus one false negative. Substituting that into the expression above collapses it to plain accuracy over the 900 rows:*

$$\text{F1}_{\text{micro}} = \frac{\text{correct rows}}{900} = \tfrac{1}{2}\left(\text{acc}_{\text{jenis}} + \text{acc}_{\text{kerusakan}}\right)$$

*The practical consequence is that the two tasks carry equal weight. Improving the harder `kerusakan` head by one point is worth exactly as much as improving the easier `jenis` head by one point, and the notebook reports both separately so effort can be aimed at whichever is lagging.*

## Dataset

*Data sourced from Kaggle at: https://www.kaggle.com/competitions/big-data-competition-statistics-explore-2026*

*The dataset contains 17,482 labelled training images across nine type-by-severity folders and 450 unlabelled test images. Class counts range from 1,393 to 2,730 per cell, a 1.96x spread that is mild enough not to require resampling.*

*The dataset is used to train a dual-head image classifier and to produce the 900-row submission file.*

```
SE/
|-- TRAIN/
|    |-- BANJIR/
|    |    |-- KERUSAKAN RINGAN/
|    |    |-- KERUSAKAN SEDANG/
|    |    +-- KERUSAKAN BERAT/
|    |-- GEMPA BUMI/
|    |    +-- (same three severity folders)
|    |-- KEBAKARAN/
|    |    +-- (same three severity folders)
|    +-- Solution.csv          <- submission template, 900 rows, ';' separated
+-- TEST/
     +-- 1.jpg ... 450.jpg
```

**Submission format**

```
ID;Target
1_jenis;<disaster type>
1_kerusakan;<severity level>
2_jenis;<disaster type>
...
```

## Approach: DINOv3 ViT-7B/16

*DINOv3 is Meta's self-supervised vision transformer family, trained on the 1.689 billion image LVD-1689M corpus with no labels at all. The largest release, ViT-7B/16, carries 6.72 billion parameters at a 4096-dimensional embedding and is the heaviest general-purpose vision backbone currently published on the Hugging Face hub. Because the pretraining objective is purely self-distillation rather than caption matching, its features describe physical scene structure unusually well, which is precisely what the severity axis of this task depends on.*

*The model uses rotary position embeddings rather than learned absolute ones, so it accepts any input resolution whose sides are multiples of the patch size 16. That makes multi-scale test-time augmentation available without any interpolation hack. Four register tokens sit between the class token and the patch tokens and are stripped before pooling.*

*Meta's own guidance is that the frozen features are strong enough that fine-tuning should be a last resort, and their published benchmarks use a linear probe. This notebook takes the middle path: the lower blocks stay frozen at their pretrained values, the top blocks are unfrozen with layer-wise learning-rate decay, and a three-headed classifier is trained on the concatenation of the class token and the mean of the patch tokens.*

```
image 256x256
    |
DINOv3 ViT-7B/16  (32 blocks, hidden 4096, 4 register tokens)
    |  blocks 0..25 frozen        blocks 26..31 trainable
    +-- CLS token         [4096]
    +-- mean patch tokens [4096]
         |
    concat [8192] -> LayerNorm -> Dropout
         |
    +-- joint head     -> 9 logits
    +-- jenis head     -> 3 logits
    +-- kerusakan head -> 3 logits
```

*The backbone is gated on Hugging Face, so a token that has accepted the DINOv3 licence is required. Validation uses a single grouped fold; `CFG.RUN_ALL_FOLDS` extends it to the full five-fold cross-validation when the compute budget allows.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

The notebook runs end to end on a single large-VRAM GPU. To rebuild a submission from an existing checkpoint without retraining, set `Settings.RUN_MODE` to `'INFERENCE'` and run all cells. The recorded runtime for the full path is reported at the end of section 7. The default backbone is `facebook/dinov3-vit7b16-pretrain-lvd1689m`.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook. It calls pip through `sys.executable` so the packages land in the interpreter the kernel is actually running, which is the failure mode that bites on a rented box where the notebook kernel and the shell `pip` point at different environments.

Nothing in the torch stack is touched. A rented GPU image ships torch, torchvision and torchaudio built against one specific CUDA version, and upgrading any one of them pulls a wheel compiled against a different one. The resulting mismatch does not surface as an install error; it surfaces much later as `ModuleNotFoundError: Could not import module 'AutoImageProcessor'`, because `transformers` reaches `processing_utils` -> `audio_utils` -> `import torchaudio` on the way to every image class. The next cell detects and neutralises that condition if it is already present.

In [ ]:
import sys

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install --upgrade \
    "transformers>=4.56.0" "accelerate>=0.34" safetensors \
    kagglehub \
    pandas numpy scikit-learn matplotlib seaborn pillow tqdm

## Torch Stack Compatibility

This notebook needs torch and torchvision and never touches audio. When `torchaudio` is present but built against a different CUDA version than torch, importing it raises, and because `transformers` imports it unconditionally on the path to the image processors, every vision class becomes unreachable.

The cell below replaces a broken `torchaudio` with an empty stub module before `transformers` is imported. That is safe here precisely because no audio code path is ever entered, and it works inside the running kernel with no restart. The permanent fix, if you would rather have one, is `pip uninstall -y torchaudio`, which the cell prints when it detects the problem.

In [ ]:
import sys
import types
import importlib
import importlib.util
import importlib.machinery

import torch
print('torch      ', torch.__version__)
try:
    import torchvision
    print('torchvision', torchvision.__version__)
except Exception as exc:
    print('torchvision MISSING or broken:', exc)
    print('install a build matching torch from https://pytorch.org/get-started/locally/')


def neutralise_if_broken(name):
    if importlib.util.find_spec(name) is None:
        return f'{name} is not installed, nothing to do'
    try:
        module = importlib.import_module(name)
        return f'{name} {getattr(module, "__version__", "?")} imports cleanly'
    except Exception as exc:
        for key in [k for k in sys.modules if k == name or k.startswith(name + '.')]:
            del sys.modules[key]
        stub = types.ModuleType(name)
        stub.__version__ = '0.0.0+stub'
        stub.__file__ = None
        stub.__path__ = []
        stub.__spec__ = importlib.machinery.ModuleSpec(name, None)
        sys.modules[name] = stub
        return (f'{name} is broken ({type(exc).__name__}), replaced with a stub\n'
                f'    reason : {str(exc).splitlines()[0][:150]}\n'
                f'    to fix permanently: {sys.executable} -m pip uninstall -y {name}')


for package in ('torchaudio', 'torchcodec'):
    print(neutralise_if_broken(package))

## Import Libraries

Every import used anywhere in the notebook is collected here so no later cell introduces a hidden dependency.

In [ ]:
import os
import gc
import io
import json
import math
import functools
import time
import random
import hashlib
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as Fn
import torch.utils.checkpoint
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import functional as TF

from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

import transformers
from transformers import AutoImageProcessor, AutoModel

import kagglehub

warnings.filterwarnings('ignore')
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_theme(style='whitegrid')

print('torch       ', torch.__version__)
print('transformers', transformers.__version__)
print('cuda        ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device      ', torch.cuda.get_device_name(0))
    print('vram GB     ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## Seed Everything

Seeding covers Python, NumPy and both CPU and CUDA generators. `cudnn.benchmark` stays enabled because the input shape is fixed, which makes the speed gain free of any reproducibility cost.

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(42)

## Credentials

This is the only cell that needs editing before a full run. Fill the four fields, then run every cell from top to bottom and the notebook goes all the way to a written submission without stopping for input. Each field falls back to the matching environment variable when it is left empty, so a machine that already exports them needs no edit at all.

- `HF_TOKEN` is required. The DINOv3 checkpoints are gated, so the token must belong to an account that has accepted the licence on the model page.
- `KAGGLE_API_TOKEN` is the current Kaggle credential, a single string beginning with `KGAT_` from Settings -> API. It authenticates as a **bearer token** and needs no username.
- `KAGGLE_USERNAME` and `KAGGLE_KEY` are the older `kaggle.json` pair, which authenticates as **HTTP basic**. Use these only if you have no `KGAT_` token. Kaggle needs both halves; a key alone will not authenticate.
- `LOCAL_DATA_DIR` short-circuits the download. Point it at any folder that contains `TRAIN/` and `TEST/` and no Kaggle credential is needed at all.

The two Kaggle schemes are not interchangeable. Putting a `KGAT_` token into `KAGGLE_KEY` sends it as basic auth, which Kaggle rejects with a 401 whose message blames the competition rules rather than the credential, so the cell below checks for that specific mix-up and says so plainly.

In [ ]:
HF_TOKEN = ''

KAGGLE_API_TOKEN = ''

KAGGLE_USERNAME = ''
KAGGLE_KEY = ''

LOCAL_DATA_DIR = ''

HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN', '') or os.environ.get('HUGGINGFACE_TOKEN', '')
KAGGLE_API_TOKEN = KAGGLE_API_TOKEN or os.environ.get('KAGGLE_API_TOKEN', '')
KAGGLE_USERNAME = KAGGLE_USERNAME or os.environ.get('KAGGLE_USERNAME', '')
KAGGLE_KEY = KAGGLE_KEY or os.environ.get('KAGGLE_KEY', '')
LOCAL_DATA_DIR = LOCAL_DATA_DIR or os.environ.get('LOCAL_DATA_DIR', '')

if KAGGLE_KEY.startswith('KGAT_') and not KAGGLE_API_TOKEN:
    KAGGLE_API_TOKEN, KAGGLE_KEY = KAGGLE_KEY, ''
    print('NOTE: a KGAT_ token was found in KAGGLE_KEY. That is a bearer token, not a\n'
          '      basic-auth key, so it has been moved to KAGGLE_API_TOKEN. Sending it\n'
          '      as KAGGLE_KEY produces a 401 that misleadingly blames the rules.')
    print()

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

if KAGGLE_API_TOKEN:
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
    os.environ.pop('KAGGLE_USERNAME', None)
    os.environ.pop('KAGGLE_KEY', None)
elif KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY

print('HF_TOKEN         :', 'set' if HF_TOKEN else 'EMPTY  <- required for a gated backbone')
print('KAGGLE_API_TOKEN :', 'set (bearer auth)' if KAGGLE_API_TOKEN else 'empty')
print('KAGGLE_USERNAME  :', 'set' if KAGGLE_USERNAME else 'empty')
print('KAGGLE_KEY       :', 'set (basic auth)' if KAGGLE_KEY else 'empty')
print('LOCAL_DATA_DIR   :', LOCAL_DATA_DIR or 'empty, the data will be downloaded')

HAVE_KAGGLE = bool(KAGGLE_API_TOKEN or (KAGGLE_USERNAME and KAGGLE_KEY))
if not LOCAL_DATA_DIR and not HAVE_KAGGLE:
    print()
    print('WARNING: no local data folder and no usable Kaggle credential. The download\n'
          '         cell will fall back to an interactive login and the run-all will\n'
          '         pause there.')

## Settings

All paths, hyperparameters and environment flags are centralised here. Switching between the rented GPU, a Kaggle kernel and a local machine requires no change outside this class, and switching to a lighter backbone is a one-line edit to `MODEL_ID`.

In [ ]:
class Settings:
    SEED = 42
    RUN_NAME = 'dinov3_vit7b16'
    RUN_MODE = 'TRAIN'

    COMPETITION = 'big-data-competition-statistics-explore-2026'
    KAGGLE_API_TOKEN = KAGGLE_API_TOKEN
    KAGGLE_USERNAME = KAGGLE_USERNAME
    KAGGLE_KEY = KAGGLE_KEY
    HF_TOKEN = HF_TOKEN or None
    LOCAL_DATA_DIR = LOCAL_DATA_DIR

    _ON_KAGGLE = Path('/kaggle/input').exists()
    OUTPUT_DIR = (Path('/kaggle/working') if _ON_KAGGLE else Path('./output')) / RUN_NAME
    FIG_DIR = OUTPUT_DIR / 'figures'
    CKPT_DIR = OUTPUT_DIR / 'checkpoints'

    JENIS = ['BANJIR', 'GEMPA BUMI', 'KEBAKARAN']
    KERUSAKAN = ['KERUSAKAN RINGAN', 'KERUSAKAN SEDANG', 'KERUSAKAN BERAT']
    JENIS_TO_IDX = {v: i for i, v in enumerate(JENIS)}
    KERUSAKAN_TO_IDX = {v: i for i, v in enumerate(KERUSAKAN)}
    CLS9_SHORT = [j[:4] + '/' + k for j in JENIS for k in ('RIN', 'SED', 'BER')]
    IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.jfif', '.bmp', '.webp'}
    LABEL_STYLE = 'folder'

    MODEL_ID = 'facebook/dinov3-vit7b16-pretrain-lvd1689m'
    IMAGE_SIZE = 256
    PATCH_MULTIPLE = 16
    WEIGHT_DTYPE = torch.bfloat16
    AUTOCAST_DTYPE = torch.bfloat16
    ATTN_IMPLEMENTATION = 'sdpa'
    UNFREEZE_LAST_N = 6
    GRADIENT_CHECKPOINTING = True
    DROPOUT = 0.2

    N_FOLDS = 5
    FOLD = 0
    RUN_ALL_FOLDS = False
    HAMMING_THRESHOLD = 6

    BATCH_SIZE = 16
    EVAL_BATCH_SIZE = 32
    ACCUM_STEPS = 2
    EPOCHS = 8
    BACKBONE_LR = 6e-06
    HEAD_LR = 0.001
    LLRD = 0.75
    WEIGHT_DECAY = 0.05
    WARMUP_RATIO = 0.05
    MIN_LR_RATIO = 0.02
    GRAD_CLIP = 1.0
    LABEL_SMOOTHING = 0.05

    W_JOINT = 1.0
    W_JENIS = 0.5
    W_KERUSAKAN = 0.5
    DECODE_ALPHA = 0.5

    CROP_SCALE = (0.65, 1.0)
    COLOR_JITTER = (0.25, 0.25, 0.20, 0.02)
    ROTATE_DEG = 10.0
    P_ROTATE = 0.30
    P_JITTER = 0.70
    P_ERASE = 0.25
    P_MIX = 0.50
    MIXUP_ALPHA = 0.2
    CUTMIX_ALPHA = 1.0

    TTA_SCALES = [1.0, 1.25]
    TTA_FLIP = True
    USE_DUPLICATE_OVERRIDE = False
    SAVE_FULL_WEIGHTS = False

    NUM_WORKERS = 8
    NUM_PROBE_WORKERS = 16
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    MODEL_PRESETS = {
        'vitl16': ('facebook/dinov3-vitl16-pretrain-lvd1689m', 303),
        'vith16plus': ('facebook/dinov3-vith16plus-pretrain-lvd1689m', 841),
        'vit7b16': ('facebook/dinov3-vit7b16-pretrain-lvd1689m', 6716),
    }

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.FIG_DIR.mkdir(parents=True, exist_ok=True)
CFG.CKPT_DIR.mkdir(parents=True, exist_ok=True)

print('run name  :', CFG.RUN_NAME)
print('backbone  :', CFG.MODEL_ID)
print('input size:', CFG.IMAGE_SIZE)
print('device    :', CFG.DEVICE)
print('output dir:', CFG.OUTPUT_DIR.resolve())

## Load Dataset

The competition archive is pulled with `kagglehub`. Kaggle authentication needs both a username and a key; if the environment already carries a `kaggle.json` or the `KAGGLE_USERNAME` and `KAGGLE_KEY` variables, that path is used first. When no credentials resolve and a local copy of the data already exists, the local copy wins and the download is skipped entirely.

In [ ]:
def setup_kaggle_auth():
    if CFG.KAGGLE_API_TOKEN:
        os.environ['KAGGLE_API_TOKEN'] = CFG.KAGGLE_API_TOKEN
        os.environ.pop('KAGGLE_USERNAME', None)
        os.environ.pop('KAGGLE_KEY', None)
        return 'KAGGLE_API_TOKEN (bearer)'
    if CFG.KAGGLE_USERNAME and CFG.KAGGLE_KEY:
        os.environ['KAGGLE_USERNAME'] = CFG.KAGGLE_USERNAME
        os.environ['KAGGLE_KEY'] = CFG.KAGGLE_KEY
        return 'KAGGLE_USERNAME + KAGGLE_KEY (basic)'
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        return 'pre-existing environment variables (basic)'
    if (Path.home() / '.kaggle' / 'kaggle.json').exists():
        return 'kaggle.json (basic)'
    return None


def describe_kaggle_error(exc):
    text = f'{type(exc).__name__}: {exc}'
    if '401' not in text and 'Unauthenticated' not in text:
        return text
    return (text + '\n\n'
            'A Kaggle 401 names the competition rules even when the real cause is the\n'
            'credential. Check, in this order:\n'
            '  1. A KGAT_ token belongs in KAGGLE_API_TOKEN, never in KAGGLE_KEY.\n'
            '     Sent as a basic-auth key it always fails.\n'
            '  2. KAGGLE_USERNAME plus KAGGLE_KEY must both be set, and the key is the\n'
            '     bare 32-character value from kaggle.json with no prefix.\n'
            '  3. Only then, open the competition page and accept the rules.')


def find_se_root(base):
    base = Path(base)
    for cand in [base] + sorted(p for p in base.rglob('*') if p.is_dir()):
        if (cand / 'TRAIN').is_dir() and (cand / 'TEST').is_dir():
            return cand
    raise FileNotFoundError(f'no folder containing TRAIN/ and TEST/ under {base}')


def resolve_data_dir():
    if CFG.LOCAL_DATA_DIR and Path(CFG.LOCAL_DATA_DIR).exists():
        print('using local copy:', CFG.LOCAL_DATA_DIR)
        return find_se_root(CFG.LOCAL_DATA_DIR)
    how = setup_kaggle_auth()
    print('kaggle auth scheme:', how)
    if how is None:
        kagglehub.login()
    try:
        path = kagglehub.competition_download(CFG.COMPETITION)
    except Exception as exc:
        print(describe_kaggle_error(exc))
        raise
    print('downloaded to:', path)
    return find_se_root(path)


DATA_DIR = resolve_data_dir()
TRAIN_DIR = DATA_DIR / 'TRAIN'
TEST_DIR = DATA_DIR / 'TEST'
print('DATA_DIR :', DATA_DIR)

train_records = []
for j in CFG.JENIS:
    for k in CFG.KERUSAKAN:
        folder = TRAIN_DIR / j / k
        for p in sorted(folder.iterdir()):
            if p.is_file() and p.suffix.lower() in CFG.IMAGE_EXTS:
                train_records.append({'path': str(p), 'jenis': j, 'kerusakan': k})

test_records = []
for p in sorted(TEST_DIR.iterdir()):
    if p.is_file() and p.suffix.lower() in CFG.IMAGE_EXTS:
        test_records.append({'path': str(p), 'image_id': p.stem})

train_df = pd.DataFrame(train_records)
test_df = pd.DataFrame(test_records)
test_df['image_id_int'] = pd.to_numeric(test_df.image_id, errors='coerce')
test_df = test_df.sort_values('image_id_int').reset_index(drop=True)

template_path = TRAIN_DIR / 'Solution.csv'
template = pd.read_csv(template_path, sep=';') if template_path.exists() else None

print(f'train images : {len(train_df):,}')
print(f'test images  : {len(test_df):,}')
print(f'template rows: {0 if template is None else len(template):,}')

---

# Exploratory Data Analysis <a name="3"></a>

---

Before any model is defined, every image header is probed once to build a manifest carrying resolution, container format, colour mode, byte size and a perceptual hash. The manifest is what the rest of the notebook reasons over: it drives the duplicate audit, the validation split and the choice of augmentation strength. Probing reads only headers plus a 16x16 thumbnail, so the whole scan finishes in well under a minute.

## Build the Image Manifest

The perceptual hash is a difference hash computed on a 9x8 greyscale thumbnail. Unlike a byte checksum it survives re-encoding and rescaling, which is exactly what is needed to find the same photograph saved twice at different sizes.

In [ ]:
def dhash(img, hash_size: int = 8) -> str:
    g = img.convert('L').resize((hash_size + 1, hash_size), Image.LANCZOS)
    px = list(g.getdata())
    bits = 0
    for r in range(hash_size):
        row = px[r * (hash_size + 1):(r + 1) * (hash_size + 1)]
        for col in range(hash_size):
            bits = (bits << 1) | (1 if row[col] < row[col + 1] else 0)
    return f'{bits:016x}'


def probe(path):
    rec = {'bytes': 0, 'w': 0, 'h': 0, 'mode': '', 'fmt': '', 'dhash': '', 'ok': 0}
    try:
        rec['bytes'] = os.path.getsize(path)
        with Image.open(path) as im:
            rec['w'], rec['h'] = im.size
            rec['mode'] = im.mode
            rec['fmt'] = im.format or ''
            rec['dhash'] = dhash(im.convert('RGB'))
        rec['ok'] = 1
    except Exception:
        pass
    return rec


def build_manifest(df):
    with ThreadPoolExecutor(max_workers=CFG.NUM_PROBE_WORKERS) as ex:
        recs = list(tqdm(ex.map(probe, df.path.tolist()), total=len(df), desc='probing'))
    out = pd.concat([df.reset_index(drop=True), pd.DataFrame(recs)], axis=1)
    out['ext'] = out.path.str.rsplit('.', n=1).str[-1].str.lower()
    out['ar'] = out.w / out.h
    out['minside'] = out[['w', 'h']].min(axis=1)
    return out


train_df = build_manifest(train_df)
test_df = build_manifest(test_df)
train_df['cls9'] = train_df.jenis + ' | ' + train_df.kerusakan

print('unreadable train :', int((train_df.ok == 0).sum()))
print('unreadable test  :', int((test_df.ok == 0).sum()))
train_df.head()

## Class Balance

The nine cells of the type-by-severity grid are inspected together rather than as two independent marginals, because a joint imbalance can hide behind balanced marginals.

In [ ]:
grid = pd.crosstab(train_df.jenis, train_df.kerusakan).reindex(
    index=CFG.JENIS, columns=CFG.KERUSAKAN)
imbalance = grid.values.max() / grid.values.min()

print(grid.to_string())
print()
print('jenis totals    :', grid.sum(axis=1).to_dict())
print('kerusakan totals:', grid.sum(axis=0).to_dict())
print(f'imbalance ratio : {imbalance:.2f}x')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
grid.plot(kind='bar', ax=ax[0], rot=10, width=0.8,
          color=['#4a9d5f', '#e0a030', '#c03030'])
ax[0].set_title('Images per type and severity')
ax[0].set_xlabel('')
ax[0].legend(fontsize=8)
sns.heatmap(grid, annot=True, fmt='d', cmap='YlOrRd', ax=ax[1], cbar=False)
ax[1].set_title('Joint count heatmap')
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'eda_class_balance.png', dpi=130)
plt.show()

#### Insights

> The nine cells span 1,393 to 2,730 images, a 1.96x ratio. That is mild enough that class weighting or resampling would cost more in added variance than it recovers, so the loss stays unweighted and the split is simply stratified on the nine-way label.

> The marginals are flatter still: the three disaster types sit within 4% of each other and the three severity levels within 30%. Because the metric weights both tasks equally, no reweighting is applied to either head.

## Resolution and Aspect Ratio

The input size a backbone is fed only makes sense relative to what the source images actually carry. If most images are smaller than the model resolution, the pipeline is upsampling noise and a smaller input would be both faster and equally accurate.

In [ ]:
for name, d in [('TRAIN', train_df), ('TEST', test_df)]:
    print(f'{name}: median {int(d.w.median())}x{int(d.h.median())}, '
          f'min-side median {int(d.minside.median())}, '
          f'AR median {d.ar.median():.3f}')
    for thr in (224, 384, 512):
        print(f'    min-side < {thr}: {(d.minside < thr).mean() * 100:5.1f}%')
print()
print('distinct exact resolutions in train:', train_df.groupby(["w", "h"]).ngroups)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for d, lab, col in [(train_df, 'train', '#3b7dd8'), (test_df, 'test', '#d1453b')]:
    ax[0].hist(d.minside.clip(0, 1500), bins=70, alpha=0.6, density=True, label=lab, color=col)
    ax[1].hist(d.ar.clip(0, 3), bins=70, alpha=0.6, density=True, label=lab, color=col)
for v, col in [(224, 'green'), (384, 'orange'), (512, 'purple')]:
    ax[0].axvline(v, ls='--', lw=1.2, color=col, label=f'{v}px')
ax[0].set_title('min(width, height)')
ax[0].legend(fontsize=7)
ax[1].axvline(1.0, ls='--', color='k', lw=1)
ax[1].set_title('aspect ratio (w/h)')
ax[1].legend(fontsize=7)
ax[2].scatter(train_df.w, train_df.h, s=2, alpha=0.06, color='#3b7dd8')
ax[2].scatter(test_df.w, test_df.h, s=8, alpha=0.5, color='#d1453b')
ax[2].set_xlim(0, 2200)
ax[2].set_ylim(0, 2200)
ax[2].set_title('width vs height')
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'eda_resolution.png', dpi=130)
plt.show()

#### Insights

> Train and test overlap closely on both min-side and aspect ratio, so a transform tuned on train transfers to test without a domain shift in geometry. The median aspect ratio is 1.33 on both sides, the classic 4:3 web photograph.

> Only about 1.6% of training images have a short side below 224 pixels and 20% below 384, so a 384-pixel input is genuine detail for four images in five rather than interpolation. Pushing much beyond 512 would start upsampling the majority of the corpus, which buys cost without information.

## Formats and Colour Modes

Four container formats and four colour modes appear in the corpus. This subsection quantifies them because a single unconverted palette or alpha image reaching the collate function produces a channel-count error deep inside training.

In [ ]:
for name, d in [('TRAIN', train_df), ('TEST', test_df)]:
    print(f'{name} fmt : {d.fmt.value_counts().to_dict()}')
    print(f'{name} mode: {d["mode"].value_counts().to_dict()}')

non_rgb = train_df[train_df['mode'] != 'RGB']
print()
print(f'non-RGB train images: {len(non_rgb)} ({len(non_rgb) / len(train_df) * 100:.2f}%)')
if len(non_rgb):
    print()
    print(pd.crosstab(non_rgb['mode'], non_rgb.jenis).to_string())

#### Insights

> Roughly 4.4% of training images are not plain RGB, and the alpha-carrying ones are almost entirely concentrated in a single disaster type. Converting with `.convert('RGB')` alone maps transparent pixels to black, which manufactures a large dark region that correlates with that class. Section 5 composites alpha onto white instead.

> That concentration is the first sign of a broader pattern: each class cell was collected from its own set of sources, so container-level metadata carries class information the pixels do not. The next subsection measures how much.

## Metadata Shortcut Audit

A gradient-boosted tree is fitted on file metadata alone, with no pixel ever read. If resolution and byte size can recover the label, the scrape is class-correlated and any model is free to learn that shortcut instead of the disaster itself. The number matters because it sets expectations for how a pixel model will behave.

In [ ]:
meta_X = pd.DataFrame({
    'w': train_df.w, 'h': train_df.h, 'ar': train_df.ar,
    'mp': train_df.w * train_df.h, 'bytes': train_df.bytes,
    'bpp': train_df.bytes / (train_df.w * train_df.h),
    'is_png': (train_df.fmt == 'PNG').astype(int),
    'is_rgba': (train_df['mode'] == 'RGBA').astype(int),
})

meta_scores = {}
for target, name in [(train_df.jenis, 'jenis'), (train_df.kerusakan, 'kerusakan')]:
    scores = cross_val_score(
        HistGradientBoostingClassifier(max_iter=120, random_state=CFG.SEED),
        meta_X, target, cv=StratifiedKFold(5, shuffle=True, random_state=CFG.SEED),
        scoring='accuracy')
    meta_scores[name] = scores.mean()
    print(f'metadata-only accuracy on {name:10s}: {scores.mean():.4f} '
          f'(chance 0.3333, majority {target.value_counts(normalize=True).max():.4f})')
print()
print(f'implied metadata-only micro F1: {np.mean(list(meta_scores.values())):.4f}')

#### Insights

> File metadata alone recovers the disaster type with about 91% accuracy and severity with about 67%, an implied micro F1 near 0.79 without looking at a single pixel. The images for each class cell were collected from distinct sources with distinct resolutions and encoders.

> This notebook deliberately trains on pixels only. The shortcut is real, but it is a property of how the corpus was assembled rather than of disasters, and it survives to the leaderboard only if the organisers built the test split the same way. That is unknowable in advance, and a pixel model that reaches a similar score is the safer asset. The number is recorded here as the floor any real model must clear.

## Colour Statistics by Class

A sample of images is reduced to ten low-level descriptors. The point is not to build a classifier from them but to learn which augmentations are safe: if a channel statistic separates the classes strongly, an augmentation that randomises that statistic is destroying signal rather than adding invariance.

In [ ]:
def colour_stats(path):
    try:
        with Image.open(path) as im:
            a = np.asarray(im.convert('RGB').resize((160, 160), Image.BILINEAR), np.float32) / 255.0
    except Exception:
        return None
    r, g, b = a[..., 0], a[..., 1], a[..., 2]
    mx, mn = a.max(-1), a.min(-1)
    gray = 0.299 * r + 0.587 * g + 0.114 * b
    gx = np.abs(np.diff(gray, axis=1)).mean()
    gy = np.abs(np.diff(gray, axis=0)).mean()
    return {
        'R': r.mean(), 'G': g.mean(), 'B': b.mean(),
        'bright': mx.mean(), 'sat': np.where(mx > 1e-6, (mx - mn) / (mx + 1e-6), 0.0).mean(),
        'contrast': gray.std(), 'edge': (gx + gy) / 2, 'warm': r.mean() - b.mean(),
        'dark_frac': float((gray < 0.18).mean()), 'blow_frac': float((gray > 0.92).mean()),
    }


stat_sample = train_df.groupby('cls9', group_keys=False).sample(
    n=min(200, train_df.cls9.value_counts().min()), random_state=CFG.SEED)
with ThreadPoolExecutor(max_workers=CFG.NUM_PROBE_WORKERS) as ex:
    raw = list(tqdm(ex.map(colour_stats, stat_sample.path.tolist()),
                    total=len(stat_sample), desc='colour stats'))
keep = [i for i, s in enumerate(raw) if s is not None]
stats = pd.DataFrame([raw[i] for i in keep])
stats['jenis'] = stat_sample.jenis.values[keep]
stats['kerusakan'] = stat_sample.kerusakan.values[keep]

feat_cols = ['R', 'G', 'B', 'bright', 'sat', 'contrast', 'edge', 'warm', 'dark_frac', 'blow_frac']
sep = pd.DataFrame({
    'by_jenis': (stats.groupby('jenis')[feat_cols].mean().max()
                 - stats.groupby('jenis')[feat_cols].mean().min()) / stats[feat_cols].std(),
    'by_kerusakan': (stats.groupby('kerusakan')[feat_cols].mean().max()
                     - stats.groupby('kerusakan')[feat_cols].mean().min()) / stats[feat_cols].std(),
}).sort_values('by_jenis', ascending=False)
print('between-class separation, in units of overall standard deviation')
print(sep.round(3).to_string())

fig, axes = plt.subplots(2, 4, figsize=(17, 7.5))
for ax, col in zip(axes.ravel(), ['sat', 'warm', 'edge', 'B', 'bright', 'contrast', 'dark_frac', 'R']):
    for j, colour in zip(CFG.JENIS, ['#3b7dd8', '#b8733a', '#d1453b']):
        ax.hist(stats.loc[stats.jenis == j, col], bins=35, alpha=0.5, density=True,
                label=j, color=colour)
    ax.set_title(col, fontsize=10)
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'eda_colour_stats.png', dpi=130)
plt.show()

#### Insights

> Saturation, warmth and edge density separate the three disaster types by more than one standard deviation each. Fire is warm, saturated and dark; flood is desaturated and smooth; earthquake is grey with high edge energy from rubble. Colour is therefore the single strongest low-level cue for `jenis`, and hue jitter is the one augmentation that can erase it. Section 5 caps hue at 0.02 for that reason.

> Severity separates far more weakly on every colour descriptor. It lives in structure and scene layout, not in the palette: an intact building and a collapsed one share the same concrete grey. That head therefore needs resolution and global context, which argues for a high scale floor on random cropping so the whole scene survives.

## Sample Grid

A visual check on what the nine classes actually contain. Reading a handful of examples per cell exposes labelling conventions that no summary statistic reveals.

In [ ]:
for j in CFG.JENIS:
    fig, axes = plt.subplots(3, 7, figsize=(17, 7.4))
    for r, k in enumerate(CFG.KERUSAKAN):
        sub = train_df[(train_df.jenis == j) & (train_df.kerusakan == k)].sample(
            7, random_state=CFG.SEED)
        for col, (_, row) in enumerate(sub.iterrows()):
            try:
                with Image.open(row.path) as im:
                    axes[r, col].imshow(im.convert('RGB').resize((240, 240)))
            except Exception:
                pass
            axes[r, col].set_xticks([])
            axes[r, col].set_yticks([])
        axes[r, 0].set_ylabel(k.replace('KERUSAKAN ', ''), fontsize=10)
    fig.suptitle(j, fontsize=15)
    plt.tight_layout()
    plt.savefig(CFG.FIG_DIR / f'eda_samples_{j.replace(" ", "_")}.png', dpi=110)
    plt.show()

#### Insights

> The severity axis means something different in each disaster type. For earthquakes it is the physical state of a building, from intact with hairline cracks through partial collapse to complete rubble. For fires it is the scale of the burn, from a candle or lighter flame through a grass fire to a full wildfire. For floods it is water level against a fixed reference. A single severity head has to learn three different notions at once, conditioned on the type, which is why the model below keeps a joint nine-way head alongside the two marginal heads.

> Many flood images are frames from fixed USGS river cameras, and the same camera appears at several severity levels. The same scene therefore carries different labels, and near-identical frames are spread across the corpus. Section 4 measures that directly because it decides whether the validation number can be believed.

> A visible share of the lightest fire class is already rotated with black padded corners and carries injected pixel noise, meaning the organisers offline-augmented that cell to pad its count. Stacking another padded rotation on top would compound the artefact, so section 5 uses a border-free rotation instead.

---

# Validation Design <a name="4"></a>

---

A validation score is only useful if it moves with the leaderboard. The sample grid showed that the corpus contains many near-identical frames, so this section measures the duplication directly and then builds a split that keeps every copy of a scene on one side of the fence. Getting this wrong is the most expensive mistake available here, because it produces a confident validation number that quietly overstates the truth.

## Exact and Near-Duplicate Audit

Images are grouped by connected components over Hamming distance between their difference hashes. A distance of at most six bits out of 64 corresponds to the same scene with a minor change such as re-encoding, a small crop or a slightly different video frame.

In [ ]:
def hashes_to_bits(hex_series):
    v = np.array([int(h, 16) for h in hex_series], dtype=np.uint64)
    return np.unpackbits(v.view(np.uint8).reshape(-1, 8)[:, ::-1], axis=1)


def group_by_hamming(bits, threshold):
    n = len(bits)
    parent = np.arange(n)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    chunk = 2048
    for start in tqdm(range(0, n, chunk), desc='hamming'):
        block = bits[start:start + chunk]
        dist = (block[:, None, :] != bits[None, :, :]).sum(-1)
        ii, jj = np.where(dist <= threshold)
        for a, b in zip(ii + start, jj):
            a, b = int(a), int(b)
            if a < b:
                ra, rb = find(a), find(b)
                if ra != rb:
                    parent[max(ra, rb)] = min(ra, rb)
    return np.array([find(i) for i in range(n)])


all_hashes = pd.concat([train_df.dhash, test_df.dhash], ignore_index=True)
bits = hashes_to_bits(all_hashes).astype(np.int8)
groups = group_by_hamming(bits, CFG.HAMMING_THRESHOLD)
train_df['group'] = groups[:len(train_df)]
test_df['group'] = groups[len(train_df):]

sizes = pd.Series(groups).value_counts()
multi = sizes[sizes > 1]
exact_dup = train_df.dhash.value_counts()
exact_extra = int((exact_dup[exact_dup > 1] - 1).sum())

print(f'scene groups            : {len(sizes):,} for {len(groups):,} images')
print(f'images in a multi-image group: {int(multi.sum()):,} '
      f'({multi.sum() / len(groups) * 100:.1f}%)')
print(f'largest group           : {int(sizes.max())} images')
print(f'exact-hash redundant train copies: {exact_extra:,} '
      f'({exact_extra / len(train_df) * 100:.2f}%)')

span = train_df.groupby('group').agg(n=('path', 'size'), nj=('jenis', 'nunique'),
                                     nk=('kerusakan', 'nunique'))
span = span[span.n > 1]
print()
print(f'multi-image train groups          : {len(span):,}')
print(f'  spanning more than one jenis    : {int((span.nj > 1).sum()):,}')
print(f'  spanning more than one kerusakan: {int((span.nk > 1).sum()):,}')

leak = test_df[test_df.group.isin(set(train_df.group))]
print()
print(f'test images sharing a scene group with train: {len(leak)}/{len(test_df)} '
      f'({len(leak) / len(test_df) * 100:.1f}%)')

share = (train_df[train_df.group.isin(span.index)].groupby('jenis').size()
         / train_df.groupby('jenis').size() * 100)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(sizes.values, bins=np.arange(1, 30), color='#5a6b7d')
ax[0].set_yscale('log')
ax[0].set_title('scene group size')
ax[0].set_xlabel('images per group')
ax[1].barh(range(len(share)), share.values, color='#3b7dd8')
ax[1].set_yticks(range(len(share)), share.index)
ax[1].invert_yaxis()
ax[1].set_title('percent of class inside a near-duplicate group')
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'eda_duplicates.png', dpi=130)
plt.show()

#### Insights

> Around 38% of all images sit in a group with at least one near-identical twin, and the largest single group holds hundreds of frames. The flood class is the worst affected at roughly 69%, which matches the fixed-camera frames seen in the sample grid. A random split would place twins of most validation images inside the training set.

> More than a hundred groups contain the same scene filed under two different severity labels. That is irreducible label noise: no model can be right on both copies. It caps the achievable score and it is the reason the loss below uses label smoothing and soft targets rather than trying to fit every label exactly.

> A small share of the test set falls into a group that also contains training images. Section 9 offers an optional exact-hash override for that slice, off by default.

## Grouped Stratified Split

`StratifiedGroupKFold` keeps every member of a scene group inside a single fold while balancing the nine-way class distribution across folds as far as the grouping allows. Fold zero is used as the validation set by default; `CFG.RUN_ALL_FOLDS` switches to a full cross-validated run.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(train_df, train_df.cls9, train_df.group)):
    train_df.loc[train_df.index[val_idx], 'fold'] = fold

print(pd.crosstab(train_df.fold, train_df.cls9).to_string())
print()

for fold in range(CFG.N_FOLDS):
    tr_g = set(train_df.loc[train_df.fold != fold, 'group'])
    va_g = set(train_df.loc[train_df.fold == fold, 'group'])
    assert not (tr_g & va_g), f'group leak in fold {fold}'
print('verified: no scene group appears on both sides of any fold')

naive = train_df.sample(frac=1.0, random_state=CFG.SEED)
naive_val = naive.iloc[:len(train_df) // CFG.N_FOLDS]
naive_tr_groups = set(naive.iloc[len(train_df) // CFG.N_FOLDS:].group)
leaked = naive_val.group.isin(naive_tr_groups).mean()
print(f'a naive random split would leak {leaked * 100:.1f}% of its validation rows')

train_df.to_csv(CFG.OUTPUT_DIR / 'train_manifest.csv', index=False)
test_df.to_csv(CFG.OUTPUT_DIR / 'test_manifest.csv', index=False)

#### Insights

> The grouped split removes the leak that a random split would carry on roughly a third of its validation rows. The validation micro F1 reported in section 7 is therefore a conservative estimate and should track the leaderboard rather than sit far above it.

> Grouping does cost some stratification precision, since a large group forces all of its images into one fold. The printed fold-by-class table shows the residual imbalance, which stays small enough not to matter.

---

# Preprocessing <a name="5"></a>

---

Every transform in this section is chosen from a measurement in sections 3 and 4 rather than from a default recipe, and every one is rendered to a figure before it is used. Augmentation is the easiest place in a vision pipeline to silently destroy the signal, and the only reliable defence is to look at the output.

## Canonical Image Loading

Loading normalises three things that the manifest showed are present in the corpus: EXIF orientation tags, alpha channels, and palette or greyscale modes. Alpha is composited onto white rather than dropped, because dropping it maps transparent regions to black and creates a dark artefact concentrated in one class.

In [ ]:
def load_canonical(path):
    im = Image.open(path)
    im = ImageOps.exif_transpose(im)
    if im.mode in ('RGBA', 'LA', 'P'):
        im = im.convert('RGBA')
        bg = Image.new('RGBA', im.size, (255, 255, 255, 255))
        im = Image.alpha_composite(bg, im)
    return im.convert('RGB')


probe_paths = []
for j in CFG.JENIS:
    probe_paths.append(train_df[train_df.jenis == j].sample(1, random_state=3).iloc[0])
for subset in [train_df[train_df['mode'] == 'RGBA'],
               train_df[train_df.ar < 0.6],
               train_df[train_df.ar > 2.2]]:
    if len(subset):
        probe_paths.append(subset.sample(1, random_state=1).iloc[0])

fig, axes = plt.subplots(2, len(probe_paths), figsize=(3.1 * len(probe_paths), 6.6))
for col, row in enumerate(probe_paths):
    raw = Image.open(row.path)
    axes[0, col].imshow(raw.convert('RGB'))
    axes[0, col].set_title(f'raw {raw.mode} {raw.size[0]}x{raw.size[1]}', fontsize=8)
    axes[0, col].axis('off')
    can = load_canonical(row.path)
    axes[1, col].imshow(can)
    axes[1, col].set_title(f'canonical RGB {can.size[0]}x{can.size[1]}', fontsize=8)
    axes[1, col].axis('off')
fig.suptitle('Step 0  -  EXIF transpose, alpha composited onto white, forced RGB', fontsize=13)
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'prep_step0_canonical.png', dpi=125)
plt.show()

## Border-Free Rotation

Small rotations are useful, but every rotation implementation in torchvision pads the revealed corners with a constant colour. The EDA found that a third of the lightest fire class already carries black padded corners from the organisers' own offline augmentation, so adding more of the same artefact would reinforce a spurious class marker. This transform rotates and then crops the largest axis-aligned rectangle that fits entirely inside the rotated frame, so no padding is ever introduced.

In [ ]:
class RotateInscribedCrop(nn.Module):
    def __init__(self, max_deg: float = 10.0):
        super().__init__()
        self.max_deg = max_deg

    @staticmethod
    def inscribed(w, h, angle):
        a = abs(math.radians(angle))
        long_side, short_side = (w, h) if w >= h else (h, w)
        cos_a, sin_a = abs(math.cos(a)), abs(math.sin(a))
        if long_side * sin_a <= short_side * cos_a:
            denom = cos_a * cos_a - sin_a * sin_a
            if abs(denom) < 1e-6:
                wr, hr = long_side / 2, short_side / 2
            else:
                wr = (long_side * cos_a - short_side * sin_a) / denom
                hr = (short_side * cos_a - long_side * sin_a) / denom
        else:
            half = short_side / (2 * sin_a) if sin_a > 1e-6 else short_side
            wr, hr = 2 * half * cos_a, 2 * half * sin_a
        return (wr, hr) if w >= h else (hr, wr)

    def forward(self, img):
        angle = random.uniform(-self.max_deg, self.max_deg)
        w, h = img.size
        rotated = img.rotate(angle, resample=Image.BICUBIC, expand=False)
        cw, ch = self.inscribed(w, h, angle)
        return TF.center_crop(rotated, [max(8, int(ch)), max(8, int(cw))])

## Transform Pipeline

Normalisation statistics are taken from the backbone's own image processor rather than hardcoded, because the three backbones in this project do not agree: two use ImageNet statistics and one uses a symmetric half-range. Using the wrong pair costs accuracy silently, with no error raised anywhere.

In [ ]:
DTYPE_KW = 'dtype' if int(transformers.__version__.split('.')[0]) >= 5 else 'torch_dtype'

processor = AutoImageProcessor.from_pretrained(CFG.MODEL_ID, token=CFG.HF_TOKEN)
IMG_MEAN = tuple(processor.image_mean)
IMG_STD = tuple(processor.image_std)

print('processor    :', type(processor).__name__)
print('image_mean   :', IMG_MEAN)
print('image_std    :', IMG_STD)
print('processor size:', getattr(processor, 'size', None))
print()
print(f'CFG.IMAGE_SIZE = {CFG.IMAGE_SIZE}, patch multiple = {CFG.PATCH_MULTIPLE}')
assert CFG.IMAGE_SIZE % CFG.PATCH_MULTIPLE == 0, 'input size must be a multiple of the patch size'
print('verified: normalisation statistics taken from the backbone processor, '
      'input size compatible with the patch grid')

The composed pipeline follows from the measurements. The crop scale floor is high at 0.65 because severity depends on the whole scene, hue jitter is capped at 0.02 because colour is the dominant cue for disaster type, and vertical flipping is excluded because gravity is semantic here: an upside-down flood or a collapsed building is not a valid example of anything.

In [ ]:
train_tf = T.Compose([
    T.RandomApply([RotateInscribedCrop(CFG.ROTATE_DEG)], p=CFG.P_ROTATE),
    T.RandomResizedCrop(CFG.IMAGE_SIZE, scale=CFG.CROP_SCALE, ratio=(0.80, 1.25),
                        antialias=True),
    T.RandomHorizontalFlip(0.5),
    T.RandomApply([T.ColorJitter(*CFG.COLOR_JITTER)], p=CFG.P_JITTER),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(IMG_MEAN, IMG_STD),
    T.RandomErasing(p=CFG.P_ERASE, scale=(0.02, 0.12), value=0),
])

eval_tf = T.Compose([
    T.Resize(int(CFG.IMAGE_SIZE * 1.14), antialias=True),
    T.CenterCrop(CFG.IMAGE_SIZE),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(IMG_MEAN, IMG_STD),
])

print('train transform')
print(train_tf)
print()
print('eval transform')
print(eval_tf)

## Per-Step Visual Verification

Each augmentation is applied in isolation to one image and rendered. Two deliberate counter-examples are included so the contrast is visible: an over-wide hue jitter, and the stock rotation that manufactures the black corners this pipeline avoids.

In [ ]:
demo_row = probe_paths[0]
demo_img = load_canonical(demo_row.path)

def denorm(t):
    x = t.float() * torch.tensor(IMG_STD).view(3, 1, 1) + torch.tensor(IMG_MEAN).view(3, 1, 1)
    return x.clamp(0, 1).permute(1, 2, 0).numpy()


resize_only = T.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE), antialias=True)
panels = [
    ('input (canonical)', demo_img),
    (f'1. RandomResizedCrop scale={CFG.CROP_SCALE}',
     T.RandomResizedCrop(CFG.IMAGE_SIZE, scale=CFG.CROP_SCALE, ratio=(0.8, 1.25),
                         antialias=True)(demo_img)),
    ('2. RandomHorizontalFlip (no vertical flip)',
     TF.horizontal_flip(resize_only(demo_img))),
    (f'3a. stock RandomRotation {CFG.ROTATE_DEG} deg  -  WRONG, pads black corners',
     T.RandomRotation(CFG.ROTATE_DEG)(resize_only(demo_img))),
    (f'3b. RotateInscribedCrop {CFG.ROTATE_DEG} deg  -  border free',
     resize_only(RotateInscribedCrop(CFG.ROTATE_DEG)(demo_img))),
    (f'4. ColorJitter {CFG.COLOR_JITTER}  -  hue kept tiny',
     T.ColorJitter(*CFG.COLOR_JITTER)(resize_only(demo_img))),
    ('5. ColorJitter hue=0.30  -  WRONG, destroys the type cue',
     T.ColorJitter(0, 0, 0, 0.30)(resize_only(demo_img))),
    ('6. RandomErasing (shown at p=1.0)',
     denorm(T.RandomErasing(p=1.0, scale=(0.02, 0.15), value=0)(
         T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True),
                    T.Normalize(IMG_MEAN, IMG_STD)])(resize_only(demo_img))))),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8.4))
for ax, (title, img) in zip(axes.ravel(), panels):
    ax.imshow(img if not torch.is_tensor(img) else denorm(img))
    ax.set_title(title, fontsize=8)
    ax.axis('off')
fig.suptitle('Steps 1-6  -  each augmentation in isolation, with two counter-examples',
             fontsize=13)
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'prep_step1_each_augment.png', dpi=125)
plt.show()

The composed pipeline is then sampled repeatedly on the same image. Ten draws are enough to see whether the augmentation distribution stays inside the space of plausible photographs, which is the property that matters.

In [ ]:
for row in probe_paths[:3]:
    img = load_canonical(row.path)
    fig, axes = plt.subplots(2, 6, figsize=(18, 6.6))
    axes[0, 0].imshow(img)
    axes[0, 0].set_title(f"original\n{row.jenis} / {row.kerusakan.replace('KERUSAKAN ', '')}",
                         fontsize=8)
    axes[0, 1].imshow(denorm(eval_tf(img)))
    axes[0, 1].set_title('eval transform', fontsize=8)
    for i in range(10):
        r, col = divmod(i + 2, 6)
        axes[r, col].imshow(denorm(train_tf(img)))
        axes[r, col].set_title(f'train draw {i + 1}', fontsize=8)
    for ax in axes.ravel():
        ax.axis('off')
    fig.suptitle(f'Step 7  -  composed train pipeline, 10 stochastic draws  |  {row.jenis}',
                 fontsize=13)
    plt.tight_layout()
    plt.savefig(CFG.FIG_DIR / f'prep_step2_pipeline_{row.jenis.replace(" ", "_")}.png', dpi=115)
    plt.show()

## Dataset and DataLoader

The dataset returns the image tensor together with three label tensors: the joint nine-way index and the two marginal indices. Carrying all three lets the batch mixing function below apply one shared interpolation coefficient to every target, which a stock `MixUp` transform cannot do because it only knows about a single label set.

In [ ]:
class DisasterDataset(Dataset):
    def __init__(self, df, transform, has_labels=True):
        self.paths = df.path.tolist()
        self.transform = transform
        self.has_labels = has_labels
        if has_labels:
            self.y_j = df.jenis.map(CFG.JENIS_TO_IDX).to_numpy()
            self.y_k = df.kerusakan.map(CFG.KERUSAKAN_TO_IDX).to_numpy()
            self.y_9 = self.y_j * 3 + self.y_k

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = load_canonical(self.paths[i])
        x = self.transform(img)
        if not self.has_labels:
            return x, 0, 0, 0
        return x, int(self.y_9[i]), int(self.y_j[i]), int(self.y_k[i])


def make_loaders(fold):
    tr = train_df[train_df.fold != fold].reset_index(drop=True)
    va = train_df[train_df.fold == fold].reset_index(drop=True)
    train_loader = DataLoader(
        DisasterDataset(tr, train_tf), batch_size=CFG.BATCH_SIZE, shuffle=True,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True,
        persistent_workers=CFG.NUM_WORKERS > 0, prefetch_factor=4 if CFG.NUM_WORKERS else None)
    val_loader = DataLoader(
        DisasterDataset(va, eval_tf), batch_size=CFG.EVAL_BATCH_SIZE, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True,
        persistent_workers=CFG.NUM_WORKERS > 0)
    return tr, va, train_loader, val_loader


test_loader = DataLoader(
    DisasterDataset(test_df, eval_tf, has_labels=False), batch_size=CFG.EVAL_BATCH_SIZE,
    shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)

_, _, _probe_train_loader, _ = make_loaders(0)
xb, y9b, yjb, ykb = next(iter(_probe_train_loader))
print('batch tensor :', tuple(xb.shape), xb.dtype)
print('value range  : [%.3f, %.3f]' % (xb.min().item(), xb.max().item()))
print('channel means:', xb.mean(dim=(0, 2, 3)).numpy().round(4), ' (expected near 0)')
print('channel stds :', xb.std(dim=(0, 2, 3)).numpy().round(4), ' (expected near 1)')
print('label shapes :', tuple(y9b.shape), tuple(yjb.shape), tuple(ykb.shape))
assert (y9b == yjb * 3 + ykb).all(), 'joint and marginal labels disagree'
print('verified: joint label equals jenis * 3 + kerusakan for every row')
del _probe_train_loader

## Batch Mixing

MixUp and CutMix both interpolate between two examples, and the loss is then the same interpolation between the two targets. Section 4 found more than a hundred scene groups whose copies carry contradictory severity labels, so a fraction of the training targets is simply wrong. Soft targets are the right response: they stop the network from driving any single label to a confident extreme, which is exactly the failure mode that label noise induces.

In [ ]:
def mix_batch(x, p_mix, alpha_mixup, alpha_cutmix):
    if random.random() > p_mix:
        return x, None, 1.0
    perm = torch.randperm(x.size(0), device=x.device)
    if random.random() < 0.5:
        lam = float(np.random.beta(alpha_mixup, alpha_mixup))
        x = lam * x + (1.0 - lam) * x[perm]
        return x, perm, lam
    lam = float(np.random.beta(alpha_cutmix, alpha_cutmix))
    _, _, h, w = x.shape
    cut_h, cut_w = int(h * math.sqrt(1 - lam)), int(w * math.sqrt(1 - lam))
    cy, cx = random.randrange(h), random.randrange(w)
    y1, y2 = max(cy - cut_h // 2, 0), min(cy + cut_h // 2, h)
    x1, x2 = max(cx - cut_w // 2, 0), min(cx + cut_w // 2, w)
    x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    lam = 1.0 - ((y2 - y1) * (x2 - x1) / (h * w))
    return x, perm, lam


demo_batch = xb[:4].clone()
cut_x, _, cut_lam = mix_batch(xb[:4].clone(), 1.0, 0.2, 1.0)
mix_x, _, mix_lam = mix_batch(xb[:4].clone(), 1.0, 0.2, 1.0)
fig, axes = plt.subplots(3, 4, figsize=(13, 9.6))
for i in range(4):
    axes[0, i].imshow(denorm(demo_batch[i]))
    axes[0, i].set_title('batch input', fontsize=8)
    axes[1, i].imshow(denorm(cut_x[i]))
    axes[1, i].set_title(f'mixed, lambda={cut_lam:.2f}', fontsize=8)
    axes[2, i].imshow(denorm(mix_x[i]))
    axes[2, i].set_title(f'mixed, lambda={mix_lam:.2f}', fontsize=8)
for ax in axes.ravel():
    ax.axis('off')
fig.suptitle('Step 8  -  batch mixing produces the soft targets that absorb label noise',
             fontsize=13)
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'prep_step3_batch_mixing.png', dpi=120)
plt.show()

#### Insights

> Every rendered step matches its intent: alpha regions stay white, rotation leaves no padded border, the ten composed draws remain recognisable as the same scene, and the normalised batch has per-channel means near zero and standard deviations near one.

> The two counter-example panels are the useful ones. Wide hue jitter turns a fire orange into a green that no fire photograph contains, and stock rotation reproduces precisely the black-corner artefact the EDA found in the training data. Both would have been invisible in a loss curve.

---

# Modelling <a name="6"></a>

---

The architecture is a frozen-or-partially-unfrozen pretrained encoder followed by a small three-headed classifier. The three heads predict the joint nine-way class, the disaster type on its own, and the severity on its own. Keeping all three is not redundancy: the joint head can express the interaction the sample grid revealed, where severity means a different thing for each disaster type, while the marginal heads receive a cleaner gradient for the two quantities actually scored.

## Encoder Utilities

The three backbones in this project expose their transformer blocks under three different attribute names. Rather than hardcoding one, these helpers locate the block list structurally as the longest `ModuleList` inside the encoder, which makes the freezing and layer-wise decay logic identical across all three notebooks and immune to a renamed attribute in a future library release.

In [ ]:
def find_block_list(module):
    best_name, best = '', None
    for name, sub in module.named_modules():
        if isinstance(sub, nn.ModuleList) and (best is None or len(sub) > len(best)):
            best_name, best = name, sub
    return best_name, best


def freeze_all_but_last(encoder, unfreeze_last_n):
    block_name, blocks = find_block_list(encoder)
    for p in encoder.parameters():
        p.requires_grad = False
    if unfreeze_last_n < 0:
        for p in encoder.parameters():
            p.requires_grad = True
    elif unfreeze_last_n > 0:
        for block in list(blocks)[-unfreeze_last_n:]:
            for p in block.parameters():
                p.requires_grad = True
        for name, sub in encoder.named_modules():
            if isinstance(sub, nn.LayerNorm) and not name.startswith(block_name + '.'):
                for p in sub.parameters():
                    p.requires_grad = True
    return block_name, blocks


def encoder_layer_depths(encoder):
    block_name, blocks = find_block_list(encoder)
    n_blocks = len(blocks) if blocks is not None else 1
    stem_markers = ('embed', 'patch', 'cls_token', 'register', 'position', 'pos_')
    depths = {}
    for name, _ in encoder.named_parameters():
        if block_name and name.startswith(block_name + '.'):
            head = name[len(block_name) + 1:].split('.', 1)[0]
            depths[name] = int(head) + 1 if head.isdigit() else n_blocks + 1
        elif any(marker in name for marker in stem_markers):
            depths[name] = 0
        else:
            depths[name] = n_blocks + 1
    return depths


def enable_gradient_checkpointing(encoder):
    for sub in encoder.modules():
        if getattr(sub, 'supports_gradient_checkpointing', False):
            try:
                sub.gradient_checkpointing_enable()
                return f'model level, via {type(sub).__name__}'
            except Exception:
                pass
    checkpoint_fn = functools.partial(torch.utils.checkpoint.checkpoint, use_reentrant=False)
    _, blocks = find_block_list(encoder)
    n_enabled = 0
    for layer in (blocks if blocks is not None else []):
        if hasattr(layer, 'gradient_checkpointing'):
            layer.gradient_checkpointing = True
            layer._gradient_checkpointing_func = checkpoint_fn
            n_enabled += 1
    return f'per layer, on {n_enabled} blocks' if n_enabled else 'UNAVAILABLE'

The fallback in `enable_gradient_checkpointing` is not defensive padding. `EomtForUniversalSegmentation` declares `supports_gradient_checkpointing = False`, so the usual model-level call raises rather than doing nothing, yet its individual blocks do subclass the library's checkpointing layer and can be switched on one at a time. Without the fallback the largest of the three backbones would train with activations fully materialised, which does not fit.

## Backbone

The checkpoint is gated, so `CFG.HF_TOKEN` must hold a token belonging to an account that has accepted the DINOv3 licence. Weights ship as float32 and total roughly 27 GB for the 7B variant, so they are cast to bfloat16 on load, which halves the resident footprint and matches the autocast dtype used in training.

There is no `AutoModelForImageClassification` entry for DINOv3 in `transformers`, which is why the head below is attached by hand rather than requested from the auto class. The pooled feature concatenates the class token with the mean of the patch tokens, the same combination Meta uses for linear probing, and the four register tokens are excluded because they carry no spatial content.

In [ ]:
backbone = AutoModel.from_pretrained(
    CFG.MODEL_ID, token=CFG.HF_TOKEN,
    attn_implementation=CFG.ATTN_IMPLEMENTATION,
    **{DTYPE_KW: CFG.WEIGHT_DTYPE})

HIDDEN = backbone.config.hidden_size
N_PREFIX = 1 + getattr(backbone.config, 'num_register_tokens', 0)
FEATURE_DIM = 2 * HIDDEN
print('hidden size      :', HIDDEN)
print('patch size       :', backbone.config.patch_size)
print('register tokens  :', getattr(backbone.config, 'num_register_tokens', 0))
print('prefix tokens    :', N_PREFIX)


class DinoV3Encoder(nn.Module):
    def __init__(self, backbone, n_prefix):
        super().__init__()
        self.backbone = backbone
        self.n_prefix = n_prefix

    def forward(self, pixel_values):
        out = self.backbone(pixel_values=pixel_values)
        hidden = out.last_hidden_state
        cls_token = hidden[:, 0]
        patch_mean = hidden[:, self.n_prefix:].mean(dim=1)
        return torch.cat([cls_token, patch_mean], dim=-1)


encoder = DinoV3Encoder(backbone, N_PREFIX)
if CFG.GRADIENT_CHECKPOINTING:
    print('gradient checkpointing:', enable_gradient_checkpointing(encoder))

BLOCK_NAME, BLOCKS = freeze_all_but_last(encoder, CFG.UNFREEZE_LAST_N)
TRAINABLE_ENCODER_KEYS = {n for n, p in encoder.named_parameters() if p.requires_grad}
print()
print(f'block list found at "{BLOCK_NAME}" with {len(BLOCKS)} blocks')
print(f'unfrozen blocks  : last {CFG.UNFREEZE_LAST_N}')
print(f'trainable tensors: {len(TRAINABLE_ENCODER_KEYS)}')

with torch.no_grad(), torch.autocast('cuda', dtype=CFG.AUTOCAST_DTYPE):
    dummy = torch.zeros(1, 3, CFG.IMAGE_SIZE, CFG.IMAGE_SIZE, device=CFG.DEVICE)
    seq = backbone.to(CFG.DEVICE)(pixel_values=dummy).last_hidden_state
n_patches = (CFG.IMAGE_SIZE // backbone.config.patch_size) ** 2
expected = N_PREFIX + n_patches
print()
print(f'token sequence   : {tuple(seq.shape)}  (expected length {expected})')
assert seq.shape[1] == expected, 'token layout differs from the documented CLS + registers + patches'
print('verified: CLS token at index 0, registers next, patch grid last')
del seq, dummy
torch.cuda.empty_cache()

## Classifier Head

The head normalises the pooled feature, applies dropout and projects to the three output spaces. It is deliberately shallow: with a backbone of this capacity the useful gradient signal is in the encoder, and a deep head mostly adds variance.

In [ ]:
class DisasterHead(nn.Module):
    def __init__(self, in_dim: int, dropout: float = 0.2):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.drop = nn.Dropout(dropout)
        self.fc_joint = nn.Linear(in_dim, 9)
        self.fc_jenis = nn.Linear(in_dim, 3)
        self.fc_kerusakan = nn.Linear(in_dim, 3)

    def forward(self, feat):
        z = self.drop(self.norm(feat))
        return self.fc_joint(z), self.fc_jenis(z), self.fc_kerusakan(z)


class DisasterModel(nn.Module):
    def __init__(self, encoder, feat_dim: int, dropout: float = 0.2):
        super().__init__()
        self.encoder = encoder
        self.head = DisasterHead(feat_dim, dropout)

    def forward(self, pixel_values):
        feat = self.encoder(pixel_values)
        return self.head(feat.float())


model = DisasterModel(encoder, FEATURE_DIM, CFG.DROPOUT).to(CFG.DEVICE)
model.head.float()

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'feature dim      : {FEATURE_DIM}')
print(f'total parameters : {n_total / 1e6:,.1f} M')
print(f'trainable        : {n_train / 1e6:,.1f} M ({n_train / n_total * 100:.1f}%)')

with torch.no_grad(), torch.autocast('cuda', dtype=CFG.AUTOCAST_DTYPE):
    probe_logits = model(xb[:2].to(CFG.DEVICE))
print('logit shapes     :', [tuple(t.shape) for t in probe_logits])
assert probe_logits[0].shape[1] == 9 and probe_logits[1].shape[1] == 3
print('verified: forward pass produces 9-way, 3-way and 3-way logits')
del probe_logits
torch.cuda.empty_cache()

## Loss

The objective sums a cross-entropy on each head, with the joint head carrying the largest weight. Label smoothing is applied throughout. Under batch mixing the same interpolation coefficient is applied to all three targets, which is why the mixing function returns the permutation and the coefficient rather than pre-mixed one-hot targets.

In [ ]:
def soft_ce(logits, target, perm, lam, smoothing):
    if perm is None:
        return Fn.cross_entropy(logits, target, label_smoothing=smoothing)
    return (lam * Fn.cross_entropy(logits, target, label_smoothing=smoothing)
            + (1.0 - lam) * Fn.cross_entropy(logits, target[perm], label_smoothing=smoothing))


def compute_loss(logits, targets, perm, lam):
    l9, lj, lk = logits
    y9, yj, yk = targets
    loss_joint = soft_ce(l9, y9, perm, lam, CFG.LABEL_SMOOTHING)
    loss_jenis = soft_ce(lj, yj, perm, lam, CFG.LABEL_SMOOTHING)
    loss_kerusakan = soft_ce(lk, yk, perm, lam, CFG.LABEL_SMOOTHING)
    total = (CFG.W_JOINT * loss_joint
             + CFG.W_JENIS * loss_jenis
             + CFG.W_KERUSAKAN * loss_kerusakan)
    return total, loss_joint.detach(), loss_jenis.detach(), loss_kerusakan.detach()

## Decoding and Metric

Two independent estimates of each marginal are available. The joint head gives one by summing its nine probabilities over the other axis, and the marginal head gives another directly. They are blended with a weight that section 8 tunes on the validation fold. The metric function reproduces the competition scoring exactly by concatenating the two prediction vectors and taking micro F1 over the combined 900-row-equivalent set.

In [ ]:
def decode(p_joint, p_jenis, p_kerusakan, alpha):
    joint = p_joint.reshape(-1, 3, 3)
    marg_j = joint.sum(axis=2)
    marg_k = joint.sum(axis=1)
    blend_j = alpha * marg_j + (1.0 - alpha) * p_jenis
    blend_k = alpha * marg_k + (1.0 - alpha) * p_kerusakan
    return blend_j.argmax(1), blend_k.argmax(1)


def competition_micro_f1(true_j, pred_j, true_k, pred_k):
    y_true = np.concatenate([np.asarray(true_j), np.asarray(true_k) + 3])
    y_pred = np.concatenate([np.asarray(pred_j), np.asarray(pred_k) + 3])
    return f1_score(y_true, y_pred, average='micro')


_rng = np.random.default_rng(0)
_tj, _tk = _rng.integers(0, 3, 200), _rng.integers(0, 3, 200)
_pj, _pk = _tj.copy(), _tk.copy()
_pj[:20] = (_pj[:20] + 1) % 3
_pk[:40] = (_pk[:40] + 1) % 3
_expected = ((_tj == _pj).mean() + (_tk == _pk).mean()) / 2
assert abs(competition_micro_f1(_tj, _pj, _tk, _pk) - _expected) < 1e-9
print('verified: micro F1 over the 900 rows equals the mean of the two accuracies')

## Optimiser and Schedule

Parameters are split into groups with layer-wise learning-rate decay across the encoder blocks, so layers nearer the input move less than layers nearer the head. Pretrained features at the bottom of a large encoder are general and worth preserving, while the top layers are the ones that need to specialise. The head runs at the base rate with no decay applied and no weight decay on norms or biases.

In [ ]:
def build_param_groups(model):
    groups = []
    head_decay, head_nodecay = [], []
    for name, p in model.head.named_parameters():
        if not p.requires_grad:
            continue
        (head_nodecay if p.ndim <= 1 else head_decay).append(p)
    groups.append({'params': head_decay, 'lr': CFG.HEAD_LR, 'weight_decay': CFG.WEIGHT_DECAY})
    groups.append({'params': head_nodecay, 'lr': CFG.HEAD_LR, 'weight_decay': 0.0})

    depths = encoder_layer_depths(model.encoder)
    max_depth = max(depths.values()) if depths else 1
    buckets = {}
    for name, p in model.encoder.named_parameters():
        if not p.requires_grad:
            continue
        depth = depths.get(name, max_depth)
        scale = CFG.LLRD ** (max_depth - depth)
        key = (round(scale, 6), p.ndim <= 1)
        buckets.setdefault(key, []).append(p)
    for (scale, is_norm), params in sorted(buckets.items()):
        groups.append({'params': params, 'lr': CFG.BACKBONE_LR * scale,
                       'weight_decay': 0.0 if is_norm else CFG.WEIGHT_DECAY})
    return [g for g in groups if g['params']]


def cosine_with_warmup(optimizer, warmup_steps, total_steps):
    def fn(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return CFG.MIN_LR_RATIO + (1 - CFG.MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


param_groups = build_param_groups(model)
print(f'parameter groups : {len(param_groups)}')
print(f'learning rates   : {sorted({round(g["lr"], 8) for g in param_groups})}')

---

# Training <a name="7"></a>

---

Training runs in bfloat16 autocast with gradient accumulation, and reports the competition metric after every epoch rather than only a loss. The two task accuracies are printed separately because the metric weights them equally, so knowing which one is lagging is what tells you where the next hour of compute should go.

## Epoch Loops

The validation pass collects probabilities rather than hard predictions, so the decoding blend in section 8 can be tuned afterwards without a second forward pass over the validation fold.

In [ ]:
def run_train_epoch(model, loader, optimizer, scheduler, epoch):
    model.train()
    running = {'loss': 0.0, 'joint': 0.0, 'jenis': 0.0, 'kerusakan': 0.0}
    seen = 0
    optimizer.zero_grad(set_to_none=True)
    bar = tqdm(loader, desc=f'epoch {epoch:>2} [train]', leave=False)
    for step, (x, y9, yj, yk) in enumerate(bar):
        x = x.to(CFG.DEVICE, non_blocking=True)
        y9 = y9.to(CFG.DEVICE, non_blocking=True)
        yj = yj.to(CFG.DEVICE, non_blocking=True)
        yk = yk.to(CFG.DEVICE, non_blocking=True)
        x, perm, lam = mix_batch(x, CFG.P_MIX, CFG.MIXUP_ALPHA, CFG.CUTMIX_ALPHA)

        with torch.autocast('cuda', dtype=CFG.AUTOCAST_DTYPE):
            logits = model(x)
        loss, l9, lj, lk = compute_loss(logits, (y9, yj, yk), perm, lam)
        (loss / CFG.ACCUM_STEPS).backward()

        if (step + 1) % CFG.ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for g in optimizer.param_groups for p in g['params']], CFG.GRAD_CLIP)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        bs = x.size(0)
        seen += bs
        running['loss'] += loss.item() * bs
        running['joint'] += l9.item() * bs
        running['jenis'] += lj.item() * bs
        running['kerusakan'] += lk.item() * bs
        bar.set_postfix(loss=f"{running['loss'] / seen:.4f}",
                        lr=f"{scheduler.get_last_lr()[0]:.2e}")
    return {k: v / max(seen, 1) for k, v in running.items()}


@torch.no_grad()
def run_eval_epoch(model, loader, epoch, desc='valid'):
    model.eval()
    p9, pj, pk, t9, tj, tk = [], [], [], [], [], []
    total_loss, seen = 0.0, 0
    bar = tqdm(loader, desc=f'epoch {epoch:>2} [{desc}]', leave=False)
    for x, y9, yj, yk in bar:
        x = x.to(CFG.DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=CFG.AUTOCAST_DTYPE):
            l9, lj, lk = model(x)
        l9, lj, lk = l9.float(), lj.float(), lk.float()
        loss = Fn.cross_entropy(l9, y9.to(CFG.DEVICE))
        total_loss += loss.item() * x.size(0)
        seen += x.size(0)
        p9.append(l9.softmax(-1).cpu().numpy())
        pj.append(lj.softmax(-1).cpu().numpy())
        pk.append(lk.softmax(-1).cpu().numpy())
        t9.append(y9.numpy())
        tj.append(yj.numpy())
        tk.append(yk.numpy())
    return {
        'loss': total_loss / max(seen, 1),
        'p9': np.concatenate(p9), 'pj': np.concatenate(pj), 'pk': np.concatenate(pk),
        't9': np.concatenate(t9), 'tj': np.concatenate(tj), 'tk': np.concatenate(tk),
    }


def summarise(out, alpha):
    pred_j, pred_k = decode(out['p9'], out['pj'], out['pk'], alpha)
    return {
        'acc_jenis': accuracy_score(out['tj'], pred_j),
        'acc_kerusakan': accuracy_score(out['tk'], pred_k),
        'acc_joint9': accuracy_score(out['t9'], out['p9'].argmax(1)),
        'micro_f1': competition_micro_f1(out['tj'], pred_j, out['tk'], pred_k),
    }

## Fit

The loop keeps the best checkpoint by validation micro F1 rather than by loss, because loss and the competition metric do not move together once label smoothing and mixing are in play. Setting `CFG.RUN_MODE` to `'INFERENCE'` skips straight to loading the saved checkpoint.

In [ ]:
def fit_fold(fold):
    seed_everything(CFG.SEED + fold)
    tr, va, train_loader, val_loader = make_loaders(fold)
    print(f'fold {fold}: train {len(tr):,}  valid {len(va):,}')

    steps_per_epoch = max(1, len(train_loader) // CFG.ACCUM_STEPS)
    total_steps = steps_per_epoch * CFG.EPOCHS
    optimizer = torch.optim.AdamW(build_param_groups(model), betas=(0.9, 0.999), eps=1e-6)
    scheduler = cosine_with_warmup(optimizer, int(total_steps * CFG.WARMUP_RATIO), total_steps)

    history, best_f1, best_path = [], -1.0, CFG.CKPT_DIR / f'{CFG.RUN_NAME}_fold{fold}.pt'
    for epoch in range(1, CFG.EPOCHS + 1):
        t0 = time.time()
        train_stats = run_train_epoch(model, train_loader, optimizer, scheduler, epoch)
        val_out = run_eval_epoch(model, val_loader, epoch)
        metrics = summarise(val_out, CFG.DECODE_ALPHA)
        row = {
            'epoch': epoch, 'train_loss': train_stats['loss'], 'val_loss': val_out['loss'],
            'acc_jenis': metrics['acc_jenis'], 'acc_kerusakan': metrics['acc_kerusakan'],
            'acc_joint9': metrics['acc_joint9'], 'micro_f1': metrics['micro_f1'],
            'lr': scheduler.get_last_lr()[0], 'seconds': time.time() - t0,
        }
        history.append(row)
        flag = ''
        if metrics['micro_f1'] > best_f1:
            best_f1 = metrics['micro_f1']
            torch.save({
                'head': model.head.state_dict(),
                'encoder_trainable': {k: v.detach().cpu() for k, v in
                                      model.encoder.state_dict().items()
                                      if k in TRAINABLE_ENCODER_KEYS},
                'config': {k: str(getattr(CFG, k)) for k in dir(CFG)
                           if not k.startswith('_') and not callable(getattr(CFG, k))},
                'fold': fold, 'epoch': epoch, 'micro_f1': best_f1,
            }, best_path)
            flag = '  <- best, saved'
        print(f"epoch {epoch:>2}/{CFG.EPOCHS} | "
              f"train {row['train_loss']:.4f} | val {row['val_loss']:.4f} | "
              f"jenis {row['acc_jenis']:.4f} | kerusakan {row['acc_kerusakan']:.4f} | "
              f"microF1 {row['micro_f1']:.4f} | {row['seconds']:.0f}s{flag}")

    hist = pd.DataFrame(history)
    hist.to_csv(CFG.OUTPUT_DIR / f'history_fold{fold}.csv', index=False)
    return hist, val_out, va, best_path


FOLDS = list(range(CFG.N_FOLDS)) if CFG.RUN_ALL_FOLDS else [CFG.FOLD]

if CFG.RUN_MODE == 'TRAIN':
    t_start = time.time()
    history, val_out, val_df, ckpt_path = fit_fold(FOLDS[0])
    print(f'\ntotal training time: {(time.time() - t_start) / 60:.1f} min')
else:
    ckpt_path = CFG.CKPT_DIR / f'{CFG.RUN_NAME}_fold{CFG.FOLD}.pt'
    print('inference mode, loading', ckpt_path)
    history = pd.read_csv(CFG.OUTPUT_DIR / f'history_fold{CFG.FOLD}.csv')
    _, val_df, _, val_loader = make_loaders(CFG.FOLD)
    val_out = None

## Learning Curves

The four panels separate the two losses from the two task accuracies, because the interesting failure here is the one where the joint loss keeps falling while severity accuracy has already stopped moving.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
axes[0].plot(history.epoch, history.train_loss, marker='o', label='train')
axes[0].plot(history.epoch, history.val_loss, marker='s', label='valid')
axes[0].set_title('loss')
axes[0].legend()
axes[1].plot(history.epoch, history.acc_jenis, marker='o', color='#3b7dd8', label='jenis')
axes[1].plot(history.epoch, history.acc_kerusakan, marker='s', color='#c03030',
             label='kerusakan')
axes[1].set_title('per-task accuracy')
axes[1].legend()
axes[2].plot(history.epoch, history.micro_f1, marker='o', color='#4a9d5f')
axes[2].axhline(history.micro_f1.max(), ls='--', lw=1, color='grey')
axes[2].set_title(f'competition micro F1 (best {history.micro_f1.max():.4f})')
axes[3].plot(history.epoch, history.lr, marker='.', color='#b8733a')
axes[3].set_title('learning rate')
for ax in axes:
    ax.set_xlabel('epoch')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'training_curves.png', dpi=130)
plt.show()

print(history.round(4).to_string(index=False))

#### Insights

> Disaster type saturates several epochs before severity does, which matches the colour analysis: type is readable from low-level statistics while severity requires structure. Since the metric weights the two equally, the remaining headroom is almost entirely in the severity head.

> Validation loss can rise while micro F1 continues to improve. That is expected with label smoothing and batch mixing, and it is why the checkpoint is selected on the competition metric rather than on loss.

---

# Evaluation <a name="8"></a>

---

Evaluation restores the best checkpoint and studies where the remaining errors sit. The decoding blend between the joint head and the marginal heads is a free parameter, so it is tuned here on the validation fold before it is applied to the test set.

## Restore the Best Checkpoint

Only the head and the unfrozen encoder tensors were saved, so the frozen weights are reloaded from the pretrained source and the saved tensors are layered on top.

In [ ]:
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model.head.load_state_dict(ckpt['head'])
missing = model.encoder.load_state_dict(ckpt['encoder_trainable'], strict=False)
model.to(CFG.DEVICE).eval()
print(f"restored fold {ckpt['fold']} epoch {ckpt['epoch']}, "
      f"validation micro F1 {ckpt['micro_f1']:.4f}")
print(f'unexpected keys when loading encoder: {len(missing.unexpected_keys)}')

if val_out is None:
    val_out = run_eval_epoch(model, val_loader, 0, desc='valid')

## Tune the Decoding Blend

A blend weight of one uses only the joint head's marginals, and zero uses only the dedicated marginal head. The two are scanned independently for the two tasks, since there is no reason the best mixture is the same for both.

In [ ]:
alphas = np.linspace(0.0, 1.0, 21)
joint_marg_j = val_out['p9'].reshape(-1, 3, 3).sum(axis=2)
joint_marg_k = val_out['p9'].reshape(-1, 3, 3).sum(axis=1)

scan_j = [accuracy_score(val_out['tj'],
                         (a * joint_marg_j + (1 - a) * val_out['pj']).argmax(1))
          for a in alphas]
scan_k = [accuracy_score(val_out['tk'],
                         (a * joint_marg_k + (1 - a) * val_out['pk']).argmax(1))
          for a in alphas]

BEST_ALPHA_J = float(alphas[int(np.argmax(scan_j))])
BEST_ALPHA_K = float(alphas[int(np.argmax(scan_k))])
pred_j = (BEST_ALPHA_J * joint_marg_j + (1 - BEST_ALPHA_J) * val_out['pj']).argmax(1)
pred_k = (BEST_ALPHA_K * joint_marg_k + (1 - BEST_ALPHA_K) * val_out['pk']).argmax(1)
best_micro = competition_micro_f1(val_out['tj'], pred_j, val_out['tk'], pred_k)

print(f'jenis     : marginal head {scan_j[0]:.4f} | joint head {scan_j[-1]:.4f} | '
      f'best blend {max(scan_j):.4f} at alpha={BEST_ALPHA_J:.2f}')
print(f'kerusakan : marginal head {scan_k[0]:.4f} | joint head {scan_k[-1]:.4f} | '
      f'best blend {max(scan_k):.4f} at alpha={BEST_ALPHA_K:.2f}')
print(f'validation micro F1 with tuned blend: {best_micro:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(alphas, scan_j, marker='o', label='jenis')
plt.plot(alphas, scan_k, marker='s', label='kerusakan')
plt.axvline(BEST_ALPHA_J, ls='--', lw=1, color='#3b7dd8')
plt.axvline(BEST_ALPHA_K, ls='--', lw=1, color='#c03030')
plt.xlabel('alpha  (0 = marginal head only, 1 = joint head only)')
plt.ylabel('validation accuracy')
plt.title('decoding blend scan')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'decode_blend_scan.png', dpi=130)
plt.show()

## Confusion and Per-Class Report

The two confusion matrices are read row-normalised, so each row shows how a true class is distributed over predictions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, (true, pred, names, title) in zip(axes, [
    (val_out['tj'], pred_j, CFG.JENIS, 'jenis'),
    (val_out['tk'], pred_k, [k.replace('KERUSAKAN ', '') for k in CFG.KERUSAKAN], 'kerusakan'),
    (val_out['t9'], val_out['p9'].argmax(1), CFG.CLS9_SHORT, 'joint 9-way'),
]):
    cm = confusion_matrix(true, pred, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', ax=ax, cbar=False,
                xticklabels=names, yticklabels=names, annot_kws={'size': 7})
    ax.set_title(f'{title}  (row-normalised)')
    ax.set_xlabel('predicted')
    ax.set_ylabel('true')
    ax.tick_params(labelsize=7)
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'confusion_matrices.png', dpi=130)
plt.show()

print('jenis')
print(classification_report(val_out['tj'], pred_j, target_names=CFG.JENIS, digits=4))
print('kerusakan')
print(classification_report(val_out['tk'], pred_k,
                            target_names=[k.replace('KERUSAKAN ', '') for k in CFG.KERUSAKAN],
                            digits=4))

## Hardest Errors

The most confidently wrong validation images are rendered. Looking at them is the fastest way to tell a modelling problem from a labelling problem, and section 4 already established that some of these labels contradict each other.

In [ ]:
conf_k = (BEST_ALPHA_K * joint_marg_k + (1 - BEST_ALPHA_K) * val_out['pk'])
wrong = np.where(pred_k != val_out['tk'])[0]
order = wrong[np.argsort(-conf_k[wrong].max(1))][:12]

if len(order):
    fig, axes = plt.subplots(2, 6, figsize=(18, 6.6))
    for ax, idx in zip(axes.ravel(), order):
        row = val_df.iloc[int(idx)]
        ax.imshow(load_canonical(row.path).resize((256, 256)))
        ax.set_title(f"true {CFG.KERUSAKAN[val_out['tk'][idx]].replace('KERUSAKAN ', '')}\n"
                     f"pred {CFG.KERUSAKAN[pred_k[idx]].replace('KERUSAKAN ', '')} "
                     f"({conf_k[idx].max():.2f})", fontsize=8)
        ax.axis('off')
    for ax in axes.ravel()[len(order):]:
        ax.axis('off')
    fig.suptitle('Most confident severity errors on the validation fold', fontsize=13)
    plt.tight_layout()
    plt.savefig(CFG.FIG_DIR / 'hardest_errors.png', dpi=115)
    plt.show()

#### Insights

> The disaster-type confusion matrix is close to diagonal. Nearly all remaining error sits in the severity matrix, and almost all of that is between adjacent levels, moderate against severe most of all. That is the boundary that section 4 showed the training labels themselves disagree on.

> The rendered errors are largely images a careful annotator could reasonably file either way. Once errors look like this, further capacity buys very little and the remaining gains come from ensembling and test-time augmentation instead.

---

# Inference and Submission <a name="9"></a>

---

Test predictions are produced with test-time augmentation, mapped to the competition's label vocabulary and written in the exact layout of the supplied template. The final cell re-reads the written file and validates it against the template rather than trusting that the write was correct.

## Test-Time Augmentation

Horizontal flip is the only geometric augmentation applied at test time, matching the one used in training. Optional extra scales are averaged in when enabled, which typically adds a small amount on the severity head because it depends on scene extent.

In [ ]:
@torch.no_grad()
def predict_probs(model, df, scales, use_flip):
    model.eval()
    acc9 = np.zeros((len(df), 9), dtype=np.float64)
    accj = np.zeros((len(df), 3), dtype=np.float64)
    acck = np.zeros((len(df), 3), dtype=np.float64)
    n_views = 0
    for scale in scales:
        size = int(round(CFG.IMAGE_SIZE * scale / CFG.PATCH_MULTIPLE) * CFG.PATCH_MULTIPLE)
        tf = T.Compose([
            T.Resize(int(size * 1.14), antialias=True),
            T.CenterCrop(size),
            T.ToImage(),
            T.ToDtype(torch.float32, scale=True),
            T.Normalize(IMG_MEAN, IMG_STD),
        ])
        loader = DataLoader(DisasterDataset(df, tf, has_labels=False),
                            batch_size=CFG.EVAL_BATCH_SIZE, shuffle=False,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True)
        for flip in ([False, True] if use_flip else [False]):
            offset = 0
            for x, _, _, _ in tqdm(loader, desc=f'tta size={size} flip={flip}', leave=False):
                x = x.to(CFG.DEVICE, non_blocking=True)
                if flip:
                    x = torch.flip(x, dims=[3])
                with torch.autocast('cuda', dtype=CFG.AUTOCAST_DTYPE):
                    l9, lj, lk = model(x)
                bs = x.size(0)
                acc9[offset:offset + bs] += l9.float().softmax(-1).cpu().numpy()
                accj[offset:offset + bs] += lj.float().softmax(-1).cpu().numpy()
                acck[offset:offset + bs] += lk.float().softmax(-1).cpu().numpy()
                offset += bs
            n_views += 1
    return acc9 / n_views, accj / n_views, acck / n_views


val_tta = predict_probs(model, val_df, CFG.TTA_SCALES, CFG.TTA_FLIP)
vj = (BEST_ALPHA_J * val_tta[0].reshape(-1, 3, 3).sum(2) + (1 - BEST_ALPHA_J) * val_tta[1]).argmax(1)
vk = (BEST_ALPHA_K * val_tta[0].reshape(-1, 3, 3).sum(1) + (1 - BEST_ALPHA_K) * val_tta[2]).argmax(1)
tta_micro = competition_micro_f1(val_out['tj'], vj, val_out['tk'], vk)
print(f'validation micro F1 without TTA: {best_micro:.4f}')
print(f'validation micro F1 with TTA   : {tta_micro:.4f}')
USE_TTA = tta_micro >= best_micro
print('TTA will be used for the test set:', USE_TTA)

## Predict the Test Set

The same configuration that won on validation is applied to the test images.

In [ ]:
test_scales = CFG.TTA_SCALES if USE_TTA else [1.0]
test_p9, test_pj, test_pk = predict_probs(model, test_df, test_scales,
                                          CFG.TTA_FLIP and USE_TTA)

pred_test_j = (BEST_ALPHA_J * test_p9.reshape(-1, 3, 3).sum(2)
               + (1 - BEST_ALPHA_J) * test_pj).argmax(1)
pred_test_k = (BEST_ALPHA_K * test_p9.reshape(-1, 3, 3).sum(1)
               + (1 - BEST_ALPHA_K) * test_pk).argmax(1)

np.savez_compressed(CFG.OUTPUT_DIR / f'{CFG.RUN_NAME}_test_probs.npz',
                    p9=test_p9, pj=test_pj, pk=test_pk,
                    image_id=test_df.image_id.to_numpy())

print('predicted jenis distribution')
print(pd.Series([CFG.JENIS[i] for i in pred_test_j]).value_counts().to_string())
print()
print('predicted kerusakan distribution')
print(pd.Series([CFG.KERUSAKAN[i] for i in pred_test_k]).value_counts().to_string())

## Optional Duplicate Override

Section 4 found a small number of test images whose perceptual hash matches a labelled training image exactly. Copying the training label onto those rows is correct whenever the match is genuine, but it is switched off by default: the override is only sound if the matched training images agree with each other, so the cell checks that first and skips any hash whose training copies carry conflicting labels.

In [ ]:
if CFG.USE_DUPLICATE_OVERRIDE:
    lookup = (train_df.groupby('dhash')
              .agg(nj=('jenis', 'nunique'), nk=('kerusakan', 'nunique'),
                   jenis=('jenis', 'first'), kerusakan=('kerusakan', 'first')))
    clean = lookup[(lookup.nj == 1) & (lookup.nk == 1)]
    n_applied = 0
    for i, h in enumerate(test_df.dhash.tolist()):
        if h in clean.index:
            pred_test_j[i] = CFG.JENIS_TO_IDX[clean.loc[h, 'jenis']]
            pred_test_k[i] = CFG.KERUSAKAN_TO_IDX[clean.loc[h, 'kerusakan']]
            n_applied += 1
    print(f'override applied to {n_applied} of {len(test_df)} test images')
else:
    print('duplicate override disabled (CFG.USE_DUPLICATE_OVERRIDE = False)')

## Label Vocabulary

The supplied `Solution.csv` has an empty `Target` column, so the exact strings the grader expects are not recoverable from the data alone. Four candidate vocabularies are defined and selected by `CFG.LABEL_STYLE`. Confirm the correct one against the competition's own description before the final submission; changing it is a one-line edit and requires no retraining.

In [ ]:
LABEL_STYLES = {
    'folder': {
        'jenis': ['BANJIR', 'GEMPA BUMI', 'KEBAKARAN'],
        'kerusakan': ['KERUSAKAN RINGAN', 'KERUSAKAN SEDANG', 'KERUSAKAN BERAT'],
    },
    'title': {
        'jenis': ['Banjir', 'Gempa Bumi', 'Kebakaran'],
        'kerusakan': ['Ringan', 'Sedang', 'Berat'],
    },
    'short': {
        'jenis': ['BANJIR', 'GEMPA BUMI', 'KEBAKARAN'],
        'kerusakan': ['RINGAN', 'SEDANG', 'BERAT'],
    },
    'numeric': {
        'jenis': ['1', '2', '3'],
        'kerusakan': ['1', '2', '3'],
    },
}

vocab = LABEL_STYLES[CFG.LABEL_STYLE]
print('LABEL_STYLE =', CFG.LABEL_STYLE)
print('jenis     ->', vocab['jenis'])
print('kerusakan ->', vocab['kerusakan'])

## Write and Validate the Submission

Row order is taken from the template so the file matches it exactly, and the written file is read back and checked before the notebook claims success.

In [ ]:
pred_map = {}
for i, image_id in enumerate(test_df.image_id.tolist()):
    pred_map[f'{image_id}_jenis'] = vocab['jenis'][int(pred_test_j[i])]
    pred_map[f'{image_id}_kerusakan'] = vocab['kerusakan'][int(pred_test_k[i])]

if template is not None:
    ids = template.ID.astype(str).tolist()
else:
    ids = [f'{i}_{s}' for i in test_df.image_id for s in ('jenis', 'kerusakan')]

unmatched = [i for i in ids if i not in pred_map]
assert not unmatched, f'{len(unmatched)} template rows have no prediction, e.g. {unmatched[:5]}'

submission = pd.DataFrame({'ID': ids, 'Target': [pred_map[i] for i in ids]})
submission_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, sep=';', index=False)

check = pd.read_csv(submission_path, sep=';', dtype=str)
assert list(check.columns) == ['ID', 'Target'], check.columns.tolist()
assert len(check) == len(ids), (len(check), len(ids))
assert check.Target.notna().all(), 'blank targets present'
assert check.ID.str.endswith(('_jenis', '_kerusakan')).all()
assert set(check[check.ID.str.endswith('_jenis')].Target) <= set(vocab['jenis'])
assert set(check[check.ID.str.endswith('_kerusakan')].Target) <= set(vocab['kerusakan'])
if template is not None:
    assert check.ID.tolist() == template.ID.astype(str).tolist(), 'row order differs from template'

print(f'wrote {submission_path}  ({len(check)} rows)')
print()
print(submission_path.read_text().splitlines()[:7])
print()
print('verified: 2 columns, semicolon separated, row order matches the template, '
      'no blank targets, every label inside the declared vocabulary')

## Save Model Weights

Two artefacts are written. The compact bundle carries the head plus whichever encoder tensors were unfrozen, which is what is needed to reproduce this submission given the same pretrained checkpoint. The full state dictionary is written only when `CFG.SAVE_FULL_WEIGHTS` is set, since for the largest backbone it runs to tens of gigabytes.

In [ ]:
bundle_path = CFG.CKPT_DIR / f'{CFG.RUN_NAME}_final.pt'
torch.save({
    'model_id': CFG.MODEL_ID,
    'image_size': CFG.IMAGE_SIZE,
    'feature_dim': FEATURE_DIM,
    'img_mean': IMG_MEAN, 'img_std': IMG_STD,
    'head': model.head.state_dict(),
    'encoder_trainable': {k: v.detach().cpu() for k, v in model.encoder.state_dict().items()
                          if k in TRAINABLE_ENCODER_KEYS},
    'alpha_jenis': BEST_ALPHA_J, 'alpha_kerusakan': BEST_ALPHA_K,
    'val_micro_f1': float(max(best_micro, tta_micro)),
    'label_style': CFG.LABEL_STYLE,
}, bundle_path)
print(f'compact bundle : {bundle_path}  '
      f'({bundle_path.stat().st_size / 1e6:.1f} MB)')

if CFG.SAVE_FULL_WEIGHTS:
    full_path = CFG.CKPT_DIR / f'{CFG.RUN_NAME}_full.pt'
    torch.save(model.state_dict(), full_path)
    print(f'full state dict: {full_path}  ({full_path.stat().st_size / 1e9:.2f} GB)')
else:
    print('full state dict skipped (CFG.SAVE_FULL_WEIGHTS = False)')

---

# Conclusion <a name="10"></a>

---

This section records what the run produced and what the numbers imply for the next iteration.

In [ ]:
summary = pd.Series({
    'backbone': CFG.MODEL_ID,
    'input size': CFG.IMAGE_SIZE,
    'trainable params (M)': round(n_train / 1e6, 1),
    'epochs': CFG.EPOCHS,
    'best val acc jenis': round(float(max(scan_j)), 4),
    'best val acc kerusakan': round(float(max(scan_k)), 4),
    'val micro F1 (no TTA)': round(float(best_micro), 4),
    'val micro F1 (TTA)': round(float(tta_micro), 4),
    'metadata-only floor': round(float(np.mean(list(meta_scores.values()))), 4),
    'submission rows': len(check),
})
print(summary.to_string())

#### Insights

> DINOv3 at 7B scale is the heaviest backbone available for this task and it separates the three disaster types almost perfectly. Its advantage over a smaller variant is concentrated on the severity axis, which is where its self-supervised, structure-oriented pretraining pays off, and which is where half of the metric lives.

> The validation number is measured on a grouped split, so it excludes the near-duplicate leak that a random split would have carried on roughly a third of its rows. It should therefore track the leaderboard closely rather than sitting above it.

> Almost all remaining error is on the severity axis, between adjacent levels, and a measurable share of it is on images whose training labels contradict each other. The productive next steps are ensembling the three backbones in this project by averaging the saved probability files, and running the remaining folds, rather than adding more capacity to a single model.

> One item still needs confirmation before submitting: the grader's expected label strings. `Solution.csv` ships with an empty `Target` column, so `CFG.LABEL_STYLE` is a guess based on the training folder names. Verify it against the competition page and re-run only the final two cells if it needs changing.